# 15 — Character language model

This notebook trains and samples from the reusable components in `models/character/`. Here we choose a text file and experiment settings.

In [ ]:
import re

import numpy as np

from models.character import (
    NextCharacterPredictor,
    EmbedderCharacterModel,
    CharacterTokenizer,
    generate,
    make_examples,
    train,
)
from utils import PROJECT_ROOT

## Datasource

We will be using the Italian names in [`data/italian_names.txt`](../data/italian_names.txt).

Each non-empty line in the source file is one name. The next cell normalizes names to lowercase and keeps only ASCII letters, spaces, apostrophes, and hyphens. You can point `TEXT_FILE` at any UTF-8 text file with one item per line.

In [ ]:
TEXT_FILE = PROJECT_ROOT / "data/italian_names.txt"

source_names = tuple(line for line in TEXT_FILE.read_text(encoding="utf-8").splitlines() if line.strip())
normalized_names = (line.lower() for line in source_names)
phrases = tuple(name for name in normalized_names if re.fullmatch(r"[a-z '-]+", name))

print(f"source names: {len(source_names):,}")
print(f"accepted names: {len(phrases):,}")
print(f"first phrase: {phrases[0]!r}")

## Tokenizer

Build the tokenizer from the accepted names, then inspect its vocabulary and the frequency of its character tokens.

In [ ]:
characters = set("".join(phrases))
tokenizer = CharacterTokenizer(characters)
token_ids = [token_id for phrase in phrases for token_id in tokenizer.encode(phrase)]
token_counts = np.bincount(token_ids, minlength=tokenizer.size)
most_common_ids = np.argsort(token_counts)[-5:][::-1]

print(f"vocabulary: {tokenizer.size} tokens ({len(characters)} characters + 2 special tokens)")
print(f"special token IDs: <PAD>={tokenizer.pad_id}, <EOS>={tokenizer.eos_id}")
print("token IDs:")
for token_id, token in enumerate(tokenizer.tokens):
    print(f"  {token!r}: {token_id}")
print(f"text tokens: {token_counts.sum():,}")
print("most common character tokens:")
for token_id in most_common_ids:
    print(f"  {tokenizer.tokens[token_id]!r}: {token_counts[token_id]:,}")


## Models

In [ ]:
CONTEXT_LENGTH = 16
EMBEDDING_DIM = 16
HIDDEN_FEATURES = 128

SEED = 7

rng = np.random.default_rng(SEED)
embedder = EmbedderCharacterModel(
    tokenizer.size,
    context_length=CONTEXT_LENGTH,
    embedding_dim=EMBEDDING_DIM,
    hidden_features=HIDDEN_FEATURES,
    rng=rng,
)
model = NextCharacterPredictor(embedder, rng=rng)

In [ ]:
def _generate(prompt: str, max_new_tokens: int = 24) -> str:
    sample_rng = np.random.default_rng(17)
    return generate(model, tokenizer, prompt=prompt, max_new_tokens=max_new_tokens, rng=sample_rng)


# Generation with a "random" model
print(_generate("dant"))

## Train

Change the settings below, then rerun this cell to create a fresh model. `rng` is shared by initialization and minibatch sampling so the run is reproducible.

In [ ]:
STEPS = 10_000

BATCH_SIZE = 32
LEARNING_RATE = 0.01

contexts, targets = make_examples(phrases, tokenizer, context_length=CONTEXT_LENGTH)
losses = train(model, contexts, targets, steps=STEPS, batch_size=BATCH_SIZE,
               learning_rate=LEARNING_RATE, rng=rng)

print(f"examples: {len(contexts):,}")
print(f"loss: {losses[0]:.4f} -> {losses[-1]:.4f}")

## Generate

Try prompts made only of characters in the training file. Sampling may produce a different completion each time.

In [ ]:
print(_generate("dant"))
print(_generate("elis"))
print(_generate("mat"))
print(_generate("al"))
print(_generate("com"))